# Ranking peer-reviewed papers by their abstracts relevance to my research

Using the descriptions of my research tags previously elaborated on in the [elaborate-descriptions](elaborate-descriptions.ipynb) notebook, I will now rank the peer-reviewed papers by their relevance to my research using an LLM. The ranking will be based on the similarity between the abstracts of the papers and the descriptions of my research tags.

The only API used here is OpenAI's `GPT-4o-mini`, which is a large language model (LLM) that can ingest, navigate, and generate human-like text. The model will be asked to output the similarity between the abstracts and the research tag descriptions. Other imports (`os`, `json`, and `tqdm`) are standard Python libraries used for file handling and progress tracking.

In [ ]:
import openai

import os
import json
from tqdm import tqdm

# Load your articles JSON (with abstracts included)
with open('articles_with_abstracts.json') as f:
    articles = json.load(f)
    print(f"Loaded {len(articles)} articles.")

# To resume previous work (avoiding rework), load the tagged articles
# Comment this out if you want to start fresh
with open('tagged_articles_llm.json', 'r') as f:
    tagged_articles = json.load(f)
    tags = tagged_articles['tags']
    print(f"Loaded {len(tagged_articles['articles'])} tagged articles.")

count = 0
# Match the tagged articles with the original articles
for lh in tagged_articles['articles']:
    for rh in articles:
        if lh["title"] == rh["title"] and lh["author"] == rh["author"]:
            rh["tags"] = lh["tags"]
            rh["weightedSum"] = lh["weightedSum"]
            count += 1
            break
    else:
        articles.append(lh)

print(f"Matched {count} articles with tags.")

# Uncomment the following lines to load the refined tags if not loaded from the tagged articles
# with open('refined_tags.json', 'r') as f:
#     tags = json.load(f)

Loaded 202 articles.
Loaded 202 tagged articles.
Matched 202 articles with tags.


## Prompt
The prompt for the LLM will be structured as follows:

```
Rate the relevance of the following article abstract
to the tag description on a scale of 0-10.

Tag description: {tag_description}

Article abstract: {abstract}

Respond with only a single integer between 0 (not 
relevant) and 10 (highly relevant).
```

The prompt is designed to elicit a numerical response that indicates the relevance of the abstract to the research tag description. The LLM will be asked to provide a score between 0 and 10, where 0 indicates no relevance and 10 indicates high relevance. This type of rating is not possible with a simple keyword search, or even with a more complex semantic search, as it requires a nuanced understanding of the content and context of both the abstract and the research tag description. The LLM is expected to provide a more sophisticated evaluation than a simple keyword match or semantic similarity score.

In [ ]:
# Prepare OpenAI parameters
openai.api_key = os.getenv("OPENAI_API_KEY")
MODEL = "gpt-4o-mini"

def get_relevance_score(tag_desc, abstract_text):
    prompt = (
        f"Rate the relevance of the following article abstract to the tag description on a scale of 0-10.\n\n"
        f"Tag description:\n{tag_desc}\n\n"
        f"Article abstract:\n{abstract_text}\n\n"
        "Respond with only a single integer between 0 (not relevant) and 10 (highly relevant)."
    )
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return int(response.choices[0].message.content.strip())

# Assign secondary weights via LLM
for article in tqdm(articles, desc="Processing articles"):
    if "weightedSum" in article:
        continue
    article["tags"] = []
    weighted_sum = 0
    for tag in tags:
        score = get_relevance_score(tag["description"], article["abstract"])
        if score > 0:
            article["tags"].append({
                "id": tag["id"],
                "secondaryWeight": score
            })
            weighted_sum += tag["primaryWeight"] + score
    article["weightedSum"] = weighted_sum

Loaded 202 articles.
Loaded 82 tagged articles.
Matched 82 articles with tags.


Processing articles: 100%|██████████| 202/202 [30:13<00:00,  8.98s/it]


## Sorting the results

The results will be sorted based on the relevance scores assigned by the LLM. The papers with the highest scores will be considered the most relevant to my research, while those with lower scores will be considered less relevant. This ranking will help me identify which papers are most aligned with my research interests and priorities.

In [ ]:
# Sort by total weight
articles_sorted = sorted(
    [a for a in articles if "weightedSum" in a], 
    key=lambda x: x["weightedSum"], reverse=True)

# Output final JSON
output = {
    "tags": tags,
    "articles": articles_sorted
}
with open('tagged_articles_llm.json', 'w') as f:
    json.dump(output, f, indent=2)

print("LLM-scored tagged articles JSON generated at tagged_articles_llm.json")
print(f"Total articles written: {len(articles_sorted)}")

LLM-scored tagged articles JSON generated at tagged_articles_llm_2.json
Total articles written: 202


## Filter the articles to a manageable size

The top 20 articles are selected based on their weighted sum.

About 20 more articles are selected based on their top n=int weights, where n is the number of tags meeting a threshold weight. The threshold weight is dynamically adjusted to return less than 25 articles.

In [22]:
def filter_by_relevance(articles, threshold=0, n=4):
    return [
        article for article in articles 
        if len([tag for tag in article["tags"] if tag["secondaryWeight"] >= threshold]) >= n
    ]

filtered_articles = articles_sorted[:20]
for i in range(1, 11):
    test = filter_by_relevance(articles_sorted[20:], i, 6)
    print(f"Filtered articles with threshold {i}: {len(test)}")
    if len(test) < 25:
        filtered_articles += test
        break

print(f"Filtered articles: {len(filtered_articles)}")
with open('filtered_articles.json', 'w') as f:
    # remove abstracts and tags from filtered articles
    filtered_output = sorted([
        {k: v for k, v in article.items() if k not in ["abstract", "tags"]} for article in filtered_articles
    ], key=lambda x: x["author"])
    json.dump(filtered_output, f, indent=2)

Filtered articles with threshold 1: 180
Filtered articles with threshold 2: 180
Filtered articles with threshold 3: 142
Filtered articles with threshold 4: 92
Filtered articles with threshold 5: 53
Filtered articles with threshold 6: 37
Filtered articles with threshold 7: 18
Filtered articles: 38


Now, let's focus on one specific tag: socioperception.

In [3]:
socioperception_articles = sorted([(a["title"], a["author"], tag["secondaryWeight"]) for a in tagged_articles["articles"] 
 if any(tag["id"] == "socioperception" and tag.get("secondaryWeight", 0) > 7 for tag in a["tags"]) 
 for tag in a["tags"] if tag["id"] == "socioperception"], key=lambda x: x[2], reverse=True)

# Output socioperception articles in a table
print("Socioperception articles:")
for title, author, weight in socioperception_articles:
    print(f"{title} by {author} (weight: {weight})")

Socioperception articles:
Diverse Learners Participating in Regular Education "Book Clubs" by Goatley (weight: 9)
Reading as Situated Language: A Sociocognitive Perspective by Gee (weight: 8)
Bringing a Culturally Sustaining Lens to Reading Intervention by Wissman (weight: 8)
‘Draw yourself and write your name’: Material-discursive agency of names and drawings in early childhood by Giorza (weight: 8)
Abstract meanings may be more dynamic, due to their sociality: Comment on “Words as social tools: Language, sociality and inner grounding in abstract concepts” by Anna M. Borghi et al by Falandays (weight: 8)
A 50-Year Journey Through an Expanding Landscape of Literacy Research by Sailors (weight: 8)
"Rise Up!": Literacies, Lived Experiences, and Identities within an In-School "Other Space" by Wissman (weight: 8)
Introducing discourse analysis : From grammar to society by Gee (weight: 8)


Socioperception is a term that I coined to describe the process of perceiving and interpreting social information. It's not used in any
of the literature included in this notebook, but using an LLM to
rate the relevancy of the term to the abstracts of other articles
is a good way to get a sense of how well it fits into the existing
literature. This would be impossible to do with a keyword search.